In [0]:
from mlflow.models.signature import infer_signature

for run_name, modelo in modelos:
    with mlflow.start_run(run_name=run_name):
        modelo.fit(X_train, y_train)
        preds = modelo.predict(X_test)
        proba = modelo.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        auc = roc_auc_score(y_test, proba)

        mlflow.log_param("modelo", type(modelo).__name__)
        mlflow.log_params(modelo.get_params())
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("auc", auc)

        # --- Nuevo: generar la signature ---
        signature = infer_signature(X_train, modelo.predict(X_train))

        mlflow.sklearn.log_model(
            modelo,
            "model",
            signature=signature,
            input_example=X_train.iloc[:5]
        )

        print(f"{run_name} ({type(modelo).__name__}): acc={acc:.4f}  f1={f1:.4f}  auc={auc:.4f}")